# GPT-2 residual stream — token-level activations

A decoder-only model, capturing all 12 blocks at once: 12 x seq_len x 768 floats
per passage. `TokenActivationMapper` trims padding before activations ever reach
host memory, keeping only real tokens.

Downloads on first run: WikiText-2 (~5 MB) and GPT-2 weights (~500 MB).


In [ ]:
from transformers import AutoModel, AutoTokenizer

from nnact import TokenActivationMapper
from nnact.utils import WikiTextSamples, activation_loader

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 ships without a pad token
model = AutoModel.from_pretrained("gpt2")  # not ...LMHeadModel: no logits needed


In [ ]:
dataset = WikiTextSamples(tokenizer, n=512, max_length=64)
mapper = TokenActivationMapper(model)

# The residual stream: every block, plus the final layer norm.
LAYERS = [f"h.{i}" for i in range(12)] + ["ln_f"]
print(f"{len(dataset)} passages | {len(LAYERS)} layers: {LAYERS}")


In [ ]:
# Cost before committing: one sample gives the per-token shape.
probe_loader = activation_loader(dataset, batch_size=1)
probe = mapper.map(probe_loader, LAYERS[:1], progress=False)
hidden = probe.activations["h.0"].shape[-1]
per_token = hidden * len(LAYERS) * 4
print(f"{per_token / 1024:.1f} KB per real token, across {len(LAYERS)} layers")


In [ ]:
# Only real tokens reach host memory: no per-sample padding waste.
loader = activation_loader(dataset, batch_size=16)
result = mapper.map(loader, LAYERS)
total_real_tokens = result.activations["h.0"].shape[0]
print(f"{total_real_tokens} real tokens across {len(dataset)} passages")


In [ ]:
# One flat (total_real_tokens, hidden) tensor per layer.
for name, tensor in result.activations.items():
    print(f"  {name:<6} {tuple(tensor.shape)}")


In [ ]:
# Residual stream norm grows with depth - the usual GPT-2 picture.
for name, tensor in result.activations.items():
    print(f"  {name:<6} mean L2 norm = {tensor.norm(dim=-1).mean():6.2f}")
